# 暑さから「今日 何人 倒れるか」を予測する / Predict heat-illness from the heat

**Day 3 — 回帰(regression)の実習**です。Titanic は「助かる / 助からない」の **Yes/No(=分類)** でした。今度は **数そのものを当てる(=回帰)**。

**予測するもの**: その日・その都道府県で **熱中症で救急搬送された人数**。

**データの出所 / provenance**(作り物は一つもない):
- 搬送人数 = **総務省消防庁「熱中症による救急搬送状況」**(日別・都道府県別の政府公式記録)
- 気温・湿度 = **Open-Meteo 歴史アーカイブ**(ERA5 再解析 / 各国気象機関ベース)
- 期間 = 2018〜2024年の夏(5〜9月)、7都道府県(北海道・東京・愛知・大阪・**兵庫=神戸**・福岡・沖縄)

We predict a NUMBER (how many people are taken to hospital for heat illness on a given day), using real government records. This is regression — the counterpart to the Titanic yes/no.

## Step 1 — データを読み込む / Load the data

In [ ]:
import pandas as pd

url = "https://raw.githubusercontent.com/itoksk/summer-ai-materials/main/materials/data/heatstroke.csv"
df = pd.read_csv(url)
print(df.shape)          # (行数, 列数)
df.head()

## Step 2 — まず自分の目で見る / Look first

機械に予測させる前に、**自分の目で**「暑さ」と「搬送人数」の関係を見る。

- `tmax_c`: その日の最高気温 / `transported`: 搬送人数(これを予測する) / `pref_en`: 都道府県 / `severe`: 重症+死亡

In [ ]:
# 最高気温を3度ごとに区切って、平均の搬送人数を見る / mean transports per 3°C bin
df["tmax_bin"] = (df["tmax_c"] // 3 * 3).astype(int)
print(df.groupby("tmax_bin")["transported"].mean().round(1))

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))
plt.scatter(df["tmax_c"], df["transported"], s=6, alpha=0.15)
plt.xlabel("daily max temperature (°C)")
plt.ylabel("people transported (per prefecture/day)")
plt.title("Heat vs heat-illness transports")
plt.show()

**観察 / Observe**: 直線? それとも**曲がっている**? 30°Cを超えたあたりから急に跳ね上がる(=非線形)。これが今日のカギ。 / Straight line, or does it bend upward past ~30°C?

## Step 3 — 予測する前に予想を書く / Predict BEFORE you train

機械に答えさせる前に、**自分の予想**を決める。

1. 最高気温が **35°C** の日、ある県で何人くらい運ばれると思う? 数を1つ書く。
2. 気温の他に、搬送人数を左右しそうな要素は? (都道府県/月/湿度…) 1つ挙げる。

Write a number for "how many at 35°C" now. The model will reveal the real answer in a moment — no peeking.

## Step 4 — 「見たことのない夏」を当てる / Train on the past, predict an unseen summer

ズルを防ぐため、**年で分ける**。2018〜2023年で学習し、**2024年(モデルが一度も見ていない夏)** を当てさせる。これが「テストデータで予測する」ということ。

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

train = df[df["year"] < 2024]
test  = df[df["year"] == 2024]
print("train rows:", len(train), " / test rows (2024):", len(test))

# まずは一番単純に: 最高気温 だけ で直線を引く / simplest: a straight line from tmax only
lin = LinearRegression().fit(train[["tmax_c"]], train["transported"])
pred = lin.predict(test[["tmax_c"]])
print("直線モデルの平均誤差 / linear MAE:", round(mean_absolute_error(test["transported"], pred), 2), "人")
at35 = pd.DataFrame({"tmax_c": [35]})
print("35°Cの予測 / prediction at 35°C:", round(float(lin.predict(at35)[0]), 1), "人  <- Step3の予想と比べる")

**問い**: 直線は、暑い日(35°C超)の急増をうまく当てられた? たぶん**過小評価**したはず。関係が曲がっているのに、直線で引いたから。 / A straight line under-predicts the hottest days, because the real curve bends.

## Step 5 — もっと良いモデル + 他の手がかり / A better model with more clues

曲がりに強い **ランダムフォレスト**(たくさんの決定木の多数決)に、気温だけでなく **都道府県・月・湿度** も渡す。

In [ ]:
from sklearn.ensemble import RandomForestRegressor

feat = ["tmax_c", "tmean_c", "humidity_pct", "month"]
Xtr = pd.get_dummies(train[feat + ["pref_en"]], columns=["pref_en"])
Xte = pd.get_dummies(test[feat + ["pref_en"]], columns=["pref_en"]).reindex(columns=Xtr.columns, fill_value=0)

rf = RandomForestRegressor(n_estimators=200, random_state=42).fit(Xtr, train["transported"])
pred_rf = rf.predict(Xte)
print("森モデルの平均誤差 / RandomForest MAE:", round(mean_absolute_error(test["transported"], pred_rf), 2), "人")
print("(直線モデルより小さくなった? = 当たるようになった)")

## Step 6 — 答え合わせ: 実在の1日を予測する / Answer-check on a real day

2024年の **一番暑かった東京の日** を、モデルに予測させて、**実際の搬送人数**と比べる。

In [ ]:
test = test.copy()
test["pred"] = pred_rf.round(0)

# 2024年・東京で最高気温が一番高かった日 / hottest Tokyo day of 2024
tokyo = test[test["pref_en"] == "Tokyo"].sort_values("tmax_c", ascending=False)
print(tokyo[["date", "tmax_c", "transported", "pred"]].head(5).to_string(index=False))
print("\n予測(pred) と 実際(transported) は近い? どれくらいズレた? / How close was the model?")

In [ ]:
# どの手がかりを重視した? / which clues mattered most?
imp = sorted(zip(Xtr.columns, rf.feature_importances_), key=lambda t: -t[1])
print("特徴量の重要度 top8 / feature importance:")
for name, v in imp[:8]:
    print(f"  {name:18} {round(v, 3)}")
print("\nStep3で挙げた要素は入っていた? / did your guessed factor show up?")

## Step 7 — しきい値とアラート / The threshold and the alert

搬送人数は 30°C を超えると急増する。だから日本では **暑さ指数(WBGT)** が一定を超えると **熱中症警戒アラート**(環境省・気象庁)が出る。モデルが見つけた「曲がり角」は、国が人を守るために使っている線とほぼ同じ場所にある。

The model's "elbow" near 30°C is essentially the same line Japan uses to issue heat-illness alerts.

## ふりかえり / Reflection & limits

- このモデルは **都道府県庁所在地の気温** を県全体の代表として使っている。県は広い。Titanic と同じ警告: **モデルは「測ったもの」しか知らない**。
- 人口の多い県ほど搬送人数は多い。気温だけでなく **人口** も効いている — 公平に比べるなら「人口10万人あたり」にすべき?
- 夏は年々暑くなっている(気候変動)。学習した過去より暑い未来では、モデルの予測は当たる? 外れる?
- **チャレンジ**: (1) 「人口あたり」に直して県を公平に比べる (2) 自分の住む県を追加する(消防庁データは47都道府県すべてにある) (3) 「明日 アラートを出すべきか(Yes/No)」の**分類**に作り変える

**データ出典 / Sources**
- 総務省消防庁「熱中症による救急搬送状況」 https://www.fdma.go.jp/disaster/heatstroke/
- Open-Meteo Historical Weather API (ERA5) https://open-meteo.com/
- 環境省 熱中症予防情報サイト(暑さ指数WBGT) https://www.wbgt.env.go.jp/